# 03 - MACE Training
Train the message-passing MACE model with the same training setup as ACE and model-specific architecture choices.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.mace_wrapper import MACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize MACE model (shared training setup, model-specific architecture)
model = MACEWrapper(
    num_elements=120,
    r_max=5.0,
    num_radial=8,
    l_max=2,
    num_blocks=2,  # 2 layers of message passing
    node_dim=16
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)
print(f"Training on device: {trainer.device}\n")

Training on device: cuda



In [ ]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/mace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
mace_test_df = pd.DataFrame([test_metrics])
mace_test_df.to_csv("../data/mace_test_metrics.csv", index=False)

# Save the trained model state
torch.save(model.state_dict(), "../data/mace_model.pth")

metrics_df.head()

Epoch 000 | Time: 18.07s | Train E MAE: 340.92 meV/atom | Train F MAE: 208.08 meV/Å | Val E MAE: 186.88 meV/atom | Val F MAE: 81.03 meV/Å
Epoch 001 | Time: 16.24s | Train E MAE: 51.55 meV/atom | Train F MAE: 41.01 meV/Å | Val E MAE: 22.29 meV/atom | Val F MAE: 18.38 meV/Å
Epoch 002 | Time: 15.53s | Train E MAE: 9.46 meV/atom | Train F MAE: 12.41 meV/Å | Val E MAE: 3.55 meV/atom | Val F MAE: 8.50 meV/Å
Epoch 003 | Time: 15.50s | Train E MAE: 1.99 meV/atom | Train F MAE: 7.47 meV/Å | Val E MAE: 1.23 meV/atom | Val F MAE: 7.71 meV/Å
Epoch 004 | Time: 15.50s | Train E MAE: 0.73 meV/atom | Train F MAE: 6.67 meV/Å | Val E MAE: 1.35 meV/atom | Val F MAE: 6.47 meV/Å
Epoch 005 | Time: 15.50s | Train E MAE: 0.60 meV/atom | Train F MAE: 5.91 meV/Å | Val E MAE: 0.53 meV/atom | Val F MAE: 5.89 meV/Å
Epoch 006 | Time: 15.53s | Train E MAE: 0.39 meV/atom | Train F MAE: 5.36 meV/Å | Val E MAE: 0.58 meV/atom | Val F MAE: 5.21 meV/Å
Epoch 007 | Time: 15.52s | Train E MAE: 0.41 meV/atom | Train F MAE: 4.

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,36.388722,340.923625,208.083872,18.065400,3.282308,186.883586,81.027313
1,1,0.604774,51.554515,41.014199,16.237503,0.108368,22.290613,18.383345
2,2,0.040397,9.460490,12.414473,15.528186,0.014136,3.550696,8.498583
3,3,0.010679,1.988348,7.469849,15.499146,0.009984,1.225538,7.707475
4,4,0.008280,0.732844,6.668793,15.504822,0.007487,1.350663,6.465039


In [5]:
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 417,825
Trainable Parameters: 417,825
